# Этап 12 V1 — Research Synthesis Stage 1–11

## Исследовательский вопрос

Что Stage 1–11 действительно показали о качестве модели без `Q_B1_norm` / `Q_B2_norm`, основном ограничении текущего решения и ожидаемом information gain ещё одной architecture на тех же 47 признаках?

Это синтез уже accepted evidence. Он не меняет feature contract, split, seed или протокол Stage 1–11; не открывает final test и не создаёт новых model results.

## Чтение и проверка источников

Ниже читаются только Stage 12 evidence JSON и указанные в нём accepted summaries/results. Проверка подтверждает dataset/protocol identity, наличие источников и запрет использования final test.

In [1]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'reports').exists():
    ROOT = ROOT.parent

evidence_path = ROOT / 'reports/generated/stage12_research_synthesis_evidence_V1.json'
evidence = json.loads(evidence_path.read_text(encoding='utf-8'))
assert evidence['synthesis_only'] is True
assert evidence['final_test_used'] is False
assert evidence['protocol_identity']['features_allowed'] == 47
assert evidence['protocol_identity']['seed'] == 42
assert evidence['protocol_identity']['cv'] == '3-fold StratifiedKFold'

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

for item in evidence['source_artifacts']:
    source_path = ROOT / item['path']
    assert source_path.exists(), source_path
    assert sha256(source_path) == item['sha256'], source_path

assert len(evidence['source_notebooks']) == 11
for notebook_path in evidence['source_notebooks']:
    accepted_notebook = ROOT / notebook_path
    assert accepted_notebook.exists(), accepted_notebook

print('Проверено accepted artifacts:', len(evidence['source_artifacts']))
print('Проверено accepted notebooks:', len(evidence['source_notebooks']))
print('Final test использован:', evidence['final_test_used'])

Проверено accepted artifacts: 22
Проверено accepted notebooks: 11
Final test использован: False


## Что проверено

- Stage 4 использует accepted V2, Stage 6 — accepted V4; intermediate/rejected версии не включены.
- Каждая численная строка итоговой таблицы содержит путь к concrete accepted summary.
- `final_test_used = false` закреплён и в evidence, и в summary Stage 12.

## Общая таблица Stage 1–11

Этот шаг читает готовую CSV-таблицу, собранную только из зафиксированных summaries. Метрики не пересчитываются и не смешиваются: Gini, Recall@0.5, capacity-rescue и oracle-union остаются отдельными видами evidence.

In [2]:
import csv

table_path = ROOT / 'reports/generated/stage12_research_synthesis_table_V1.csv'
with table_path.open(encoding='utf-8', newline='') as handle:
    table = list(csv.DictReader(handle, delimiter=';'))

assert len(table) == 11
assert [row['Stage'] for row in table] == [f'Stage {n} V{v}' for n, v in [(1, 2), (2, 1), (3, 1), (4, 2), (5, 1), (6, 4), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1)]]
assert all(row['Primary evidence'].startswith('reports/summary/') for row in table)

for row in table:
    print(f"{row['Stage']}: {row['Decision']}")

Stage 1 V2: Принять baseline Gini ≈0.804
Stage 2 V1: Принять устойчивое consensus-ядро
Stage 3 V1: Перейти к diagnostic закрытых сигналов
Stage 4 V2: Stage 4 V2 accepted
Stage 5 V1: material_missing_signal
Stage 6 V4: inferior
Stage 7 V1: no_material_benefit
Stage 8 V1: no_material_benefit
Stage 9 V1: no_material_rank_complementarity
Stage 10 V1: limited_residual_model_reserve
Stage 11 V1: inferior


# FACTS / INTERPRETATION / LIMITATIONS

## FACTS

- На 47 разрешённых признаках три GBDT дают OOF Gini около 0.804; GBDT_mean в контролях Stage 7/8/11 — 0.806399.
- 805 из 1 278 deeply missed defaults составляют common blind spot, а сильное disagreement между GBDT среди deeply missed — 1.9%.
- Q_B2 лучше диагностирует blind spot, но proxy на разрешённых признаках имеет blind AUC 0.398056 против 0.723382 у oracle Q_B2; Stage 5 — `material_missing_signal`.
- TabM, stacking, FT-Transformer и RealMLP не превзошли GBDT_mean по deciding Gini; residual rescue ограничен 9/805 для FT и 15/805 для oracle-any.

## INTERPRETATION

Совокупность evidence сильнее поддерживает information/feature limitation текущих 47 признаков, чем недостаточность одной GBDT или нехватку ещё одной проверенной architecture.

## LIMITATIONS

Этот вывод не устанавливает причинность, математический ceiling Gini, невозможность будущего улучшения или temporal stability.

## Фигуры синтеза

Шесть SVG-фигур ниже созданы только по saved accepted metrics. Этот код проверяет их комплектность; сами figure files уже входят в Stage 12 evidence package.

In [3]:
figure_dir = ROOT / 'reports/figures/stage12_research_synthesis_V1'
figures = sorted(figure_dir.glob('*.svg'))
assert len(figures) == 6
for figure in figures:
    assert figure.read_text(encoding='utf-8').lstrip().startswith('<svg')
    print(figure.name)

01_карта_цепочки.svg
02_oof_gini_моделей.svg
03_blind_spot.svg
04_qb2_oracle_proxy.svg
05_residual_rescue.svg
06_выводы_и_reopen.svg


# Что мы теперь знаем

1. На текущих 47 разрешённых признаках сильный и воспроизводимый baseline существует.
2. Три GBDT и современные альтернативы дают близкую картину ограничений.
3. Существенная часть тяжёлых ошибок образует общую blind spot.
4. Diagnostic evidence Q_B2 и Stage 5 поддерживают наличие material information gap.
5. TabM, FT-Transformer, stacking, complementarity, oracle reserve и RealMLP не дали evidence, что новая architecture сама по себе решает основную проблему.

Следовательно, `CORE_MODEL_RESEARCH_STOPPED_CURRENT_47_FEATURES` означает: на текущем feature contract исследовательская ценность ещё одной model architecture стала низкой.

# Ограничения доказательств

- Нет надёжной row-level observation date; random CV/OOF не доказывает temporal stability.
- Final test не использовался для model selection.
- Recall около 69% — business reference, а не оптимизированная цель; threshold и FN/FP policy отдельно не исследовались.
- SHAP/importance не являются причинностью.
- Oracle diagnostics не являются deployable model.
- Q_B1/Q_B2 — diagnostic/reference signals, не predictors финальной модели.

# Что закрыто и что остаётся открыто

Закрыт model-only architecture search на неизменном 47-feature contract. Открыт data research: СПАРК-пилот, новые признаки, связи, динамика и другие валидные источники информации. Это не означает, что проект завершён, blind spot невозможно уменьшить или найден её причинный источник.

# Reopen conditions

Model research имеет смысл открыть снова только при одном из условий:

1. Новый валидный feature source.
2. Надёжный row-level temporal anchor.
3. Сильный независимый противоречащий результат.
4. Изменение формального business objective.
5. Новая конкретная научная гипотеза, способная изменить проектное решение.

Новый model shortlist ради количества не создаётся.

# FINAL CONCLUSION

Stage 12 V1 подготовлен как воспроизводимый synthesis accepted evidence Stage 1–11. Он поддерживает решение `CORE_MODEL_RESEARCH_STOPPED_CURRENT_47_FEATURES` в его узком смысле: ещё одна architecture на тех же 47 признаках имеет низкий ожидаемый information gain. Решение подлежит review; статус Stage 12 до принятия — `prepared_not_reviewed`.